# Train on DADA dataset

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/Thesis"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.environ["PROJECT_ROOT"] = PROJECT_ROOT
print("Project root:", PROJECT_ROOT)

Project root: /content/drive/MyDrive/Thesis


In [4]:
import os

DRIVE = '/content/drive/MyDrive/Thesis'

os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'
for v in ('KATVAD_DATA_ROOT', 'KATVAD_CACHE_ROOT', 'KATVAD_CKPT_ROOT', 'KATVAD_OUTPUT_ROOT'):
    os.makedirs(os.environ[v], exist_ok=True)

In [5]:
import os
os.environ['DADA_ORIG'] = f"{os.environ['KATVAD_DATA_ROOT']}/DADA2000Origin/DADA2000"
os.environ['PROBE']     = '/content/probe'          # VM-local NVMe, never Drive
os.makedirs(os.environ['PROBE'], exist_ok=True)

In [14]:
%%bash
apt-get -qq install -y p7zip-full
pip install -q openpyxl        # Colab usually has it; the project venv does not
ls -la "$DADA_ORIG"
df -h /content | tail -1       # how much VM disk you actually have
du -ch "$DADA_ORIG"/DADA2000.z* | tail -1   # total archive size

lrw------- 1 root root 0 Sep 15 15:11 /content/drive/MyDrive/Thesis/data/DADA2000Origin/DADA2000 -> /content/drive/.shortcut-targets-by-id/1l-TqKsT-30yRV29rW50tum-B-ZzelTQX/DADA2000
overlay         108G   22G   87G  20% /
117G	total


In [7]:
%%bash
cd "$DADA_ORIG"
7z l DADA2000.zip | head -60
echo '--- entry count ---'
7z l DADA2000.zip | tail -3


7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
1 file, 18518305241 bytes (18 GiB)

Listing archive: DADA2000.zip

--
Path = DADA2000.zip
Type = zip
Physical Size = 18518305241
Embedded Stub Size = 4
64-bit = +
Characteristics = Zip64
Total Physical Size = 125355616726
Multivolume = +
Volume Index = 5
Volumes = 6

   Date      Time    Attr         Size   Compressed  Name
------------------- ----- ------------ ------------  ------------------------
2022-11-09 00:21:22 D....            0            0  DADA2000
2022-10-29 07:52:57 D....            0            0  DADA2000/1
2022-09-23 07:22:48 D....            0            0  DADA2000/1/001
2022-10-28 12:28:19 D....            0            0  DADA2000/1/001/fixation
2020-05-22 01:25:50 ....A         1239          293  DADA2000/1/001/fixation/0001.png
2020-05-22 01:25:50 ....A         1239          293  DADA2000/1/001/fixation/0002

In [8]:
%%bash
cd "$DADA_ORIG"
# distinct top-level and second-level path prefixes
7z l -slt DADA2000.zip | grep '^Path = ' | sed 's/^Path = //' \
  | awk -F/ '{print $1"/"$2}' | sort -u | head -40

DADA2000/
DADA2000/1
DADA2000/10
DADA2000/11
DADA2000/12
DADA2000/13
DADA2000/14
DADA2000/15
DADA2000/16
DADA2000/17
DADA2000/18
DADA2000/19
DADA2000/2
DADA2000/20
DADA2000/21
DADA2000/22
DADA2000/23
DADA2000/24
DADA2000/3
DADA2000/30
DADA2000/33
DADA2000/34
DADA2000/36
DADA2000/37
DADA2000/38
DADA2000/39
DADA2000/4
DADA2000/40
DADA2000/41
DADA2000/42
DADA2000/43
DADA2000/44
DADA2000/45
DADA2000/47
DADA2000/48
DADA2000/49
DADA2000/5
DADA2000/50
DADA2000/51
DADA2000/52


In [11]:
%%bash
cd "$DADA_ORIG"
time 7z l -slt DADA2000.zip | grep -E '^(Path|Size) = ' > /content/listing.txt
wc -l /content/listing.txt

7819881 /content/listing.txt



real	1m35.339s
user	0m38.613s
sys	1m2.283s


In [12]:
%%bash
# (a) các subdir bên trong 1 clip = frame RGB nằm ở đâu?
python - <<'PY'
import collections, re
sub = collections.Counter(); size = collections.Counter(); path = None
for line in open('/content/listing.txt', encoding='utf-8', errors='replace'):
    if line.startswith('Path = '):
        path = line[7:].strip()
    elif line.startswith('Size = ') and path:
        parts = path.split('/')
        if len(parts) >= 4:
            sub[parts[3]] += 1
            size[parts[3]] += int(line[7:].strip() or 0)
        path = None
print(f"{'subdir':16s} {'#files':>10s} {'GiB':>9s}")
for k, n in sub.most_common():
    print(f'{k:16s} {n:10d} {size[k]/2**30:9.2f}')
PY

subdir               #files       GiB
fixation            1302640      3.16
maps                 651325     11.96
images               651320     94.01
seg                  651320     10.33
semantic             651320      4.04


In [13]:
%%bash
# (b) danh sách type ĐẦY ĐỦ (không head), để đối chiếu 52 type của xlsx
python - <<'PY'
import re
types = set()
for line in open('/content/listing.txt', encoding='utf-8', errors='replace'):
    if line.startswith('Path = '):
        p = line[7:].strip().split('/')
        if len(p) >= 2 and p[1].isdigit():
            types.add(int(p[1]))
print(f'{len(types)} types on disk:', sorted(types))
PY

52 types on disk: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 30, 33, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]


In [21]:
%%writefile /content/pick_probe.py
"""Pick a type-stratified probe sample from the DADA-2000 annotation.

The sheet is **detected**, not hardcoded: this workbook is a hand-maintained
export and its sheet names and header spelling are not a contract. On failure the
script prints every sheet's header so the mismatch is visible in one run.
"""
from __future__ import annotations
import argparse, json, random, re, sys
from pathlib import Path
from openpyxl import load_workbook

REQUIRED = ("type", "video", "abnormal start frame",
            "abnormal end frame", "total frames")
COL_ACCIDENT = "whether an accident occurred (1/0)"


def norm(value: object) -> str:
    """Case/whitespace-insensitive header key (the file uses NBSP in places)."""
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\xa0", " ")).strip().lower()


def find_sheet(xlsx: Path) -> tuple[list[tuple], list[str]]:
    """``(rows, header)`` for the first sheet carrying every REQUIRED column."""
    wb = load_workbook(xlsx, read_only=True, data_only=True)
    seen: list[tuple[str, list[str]]] = []
    for name in wb.sheetnames:
        it = wb[name].iter_rows(values_only=True)
        try:
            header = [norm(c) for c in next(it)]
        except StopIteration:
            seen.append((name, []))
            continue
        seen.append((name, header))
        if all(norm(col) in header for col in REQUIRED):
            print(f"sheet: {name!r}  ({len(header)} columns)")
            return list(it), header
    lines = [f"  [{n}] {h}" for n, h in seen]
    sys.exit(
        f"No sheet in {xlsx} carries all of {REQUIRED}.\nSheets found:\n"
        + "\n".join(lines)
    )


def read_rows(xlsx: Path) -> list[dict]:
    body, header = find_sheet(xlsx)
    idx = {c: header.index(norm(c)) for c in REQUIRED}
    acc = header.index(norm(COL_ACCIDENT)) if norm(COL_ACCIDENT) in header else None
    if acc is None:
        print(f"WARNING: no {COL_ACCIDENT!r} column; treating every row with a "
              "valid window as an accident row")
    out: list[dict] = []
    for raw in body:
        def cell(i: int) -> object:
            return raw[i] if i < len(raw) else None
        if acc is not None and str(cell(acc)).strip() != "1":
            continue
        try:
            typ, vid = int(cell(idx["type"])), int(cell(idx["video"]))
            start = int(cell(idx["abnormal start frame"]))
            end = int(cell(idx["abnormal end frame"]))
            total = int(cell(idx["total frames"]))
        except (TypeError, ValueError):
            continue
        if not 0 <= start < end <= total:
            continue
        out.append({"type": typ, "video": vid, "start": start,
                    "end": end, "total": total})
    return out


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--xlsx", type=Path, required=True)
    ap.add_argument("--n", type=int, default=30)
    ap.add_argument("--seed", type=int, default=2024)
    ap.add_argument("--out", type=Path, required=True)
    args = ap.parse_args(argv)

    rows = read_rows(args.xlsx)
    if not rows:
        sys.exit("No valid accident rows parsed -- check the sheet name")
    by_type: dict[int, list[dict]] = {}
    for r in rows:
        by_type.setdefault(r["type"], []).append(r)

    rng = random.Random(args.seed)
    picked: list[dict] = []
    for typ in sorted(by_type):                       # one per type first
        picked.append(rng.choice(by_type[typ]))
    rng.shuffle(picked)
    picked = picked[: args.n]
    args.out.write_text(json.dumps(picked, indent=2), encoding="utf-8")
    print(f"{len(rows)} annotated clips over {len(by_type)} types "
          f"-> picked {len(picked)}")
    for r in picked[:5]:
        print("  ", r)


if __name__ == "__main__":
    main()

Overwriting /content/pick_probe.py


In [22]:
%%bash
python /content/pick_probe.py \
  --xlsx "$DADA_ORIG/dada标注.xlsx" --n 30 --seed 2024 \
  --out /content/probe_clips.json

sheet: 'Sheet1'  (19 columns)
1945 annotated clips over 52 types -> picked 30
   {'type': 33, 'video': 1, 'start': 213, 'end': 263, 'total': 430}
   {'type': 44, 'video': 2, 'start': 160, 'end': 358, 'total': 386}
   {'type': 42, 'video': 13, 'start': 180, 'end': 360, 'total': 456}
   {'type': 47, 'video': 1, 'start': 92, 'end': 198, 'total': 382}
   {'type': 39, 'video': 16, 'start': 48, 'end': 200, 'total': 244}


In [23]:
%%bash
set -e
cd "$DADA_ORIG"
python - <<'PY' > /content/probe_patterns.txt
import json
PAT = "DADA2000/{t}/{v:03d}/images/*"     # measured layout, see section 3.1
for r in json.load(open('/content/probe_clips.json')):
    print(PAT.format(t=r['type'], v=r['video']))
PY
wc -l /content/probe_patterns.txt          # expect 30
head -3 /content/probe_patterns.txt

7z x DADA2000.zip -o"$PROBE" -y $(tr '\n' ' ' < /content/probe_patterns.txt)
du -sh "$PROBE"                            # expect ~1.5 GB (30 x ~49 MiB)
find "$PROBE" -mindepth 3 -maxdepth 3 -type d | wc -l   # expect 30

30 /content/probe_patterns.txt
DADA2000/33/001/images/*
DADA2000/44/002/images/*
DADA2000/42/013/images/*

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
1 file, 18518305241 bytes (18 GiB)

Extracting archive: DADA2000.zip
--
Path = DADA2000.zip
Type = zip
Physical Size = 18518305241
Embedded Stub Size = 4
64-bit = +
Characteristics = Zip64
Total Physical Size = 125355616726
Multivolume = +
Volume Index = 5
Volumes = 6

Everything is Ok

Files: 10323
Size:       1491283234
Compressed: 125355616726
1.5G	/content/probe
30


# P1 - Hard Gate

In [24]:
%%writefile /content/check_p1.py
"""P1: does the annotation's `total frames` match the real frame count?"""
from __future__ import annotations
import argparse, json, subprocess
from pathlib import Path

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp"}
VIDEO_SUFFIXES = {".mp4", ".avi", ".mov", ".mkv"}


def count_images(folder: Path) -> int:
    return sum(1 for p in folder.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)


def count_video_frames(path: Path) -> int:
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-count_frames", "-select_streams", "v:0",
         "-show_entries", "stream=nb_read_frames", "-of", "csv=p=0", str(path)],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    return int(out) if out.isdigit() else -1


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--clips", type=Path, required=True)
    ap.add_argument("--root", type=Path, required=True)
    ap.add_argument("--template", required=True,
                    help="path under --root, e.g. 'DADA2000/{t}/{v:03d}/images'")
    ap.add_argument("--tolerance", type=int, default=2)
    args = ap.parse_args()

    rows = json.loads(args.clips.read_text(encoding="utf-8"))
    checked = matched = missing = 0
    deltas: list[tuple[str, int, int]] = []
    for r in rows:
        rel = args.template.format(t=r["type"], v=r["video"])
        target = args.root / rel
        if target.is_dir():
            real = count_images(target)
        elif target.is_file() and target.suffix.lower() in VIDEO_SUFFIXES:
            real = count_video_frames(target)
        else:
            cand = [p for p in (args.root / rel).parent.glob(f"{Path(rel).name}*")
                    if p.suffix.lower() in VIDEO_SUFFIXES] if (args.root / rel).parent.is_dir() else []
            if len(cand) == 1:
                real = count_video_frames(cand[0])
            else:
                missing += 1
                continue
        checked += 1
        delta = real - r["total"]
        if abs(delta) <= args.tolerance:
            matched += 1
        else:
            deltas.append((rel, r["total"], real))

    print(f"checked   {checked}   missing on disk {missing}")
    if checked:
        print(f"matched within +-{args.tolerance}: {matched}/{checked} "
              f"({100 * matched / checked:.1f}%)")
    if deltas:
        print(f"\n{'clip':34s} {'annotation':>10s} {'on disk':>8s} {'delta':>7s}")
        for rel, ann, real in sorted(deltas, key=lambda d: abs(d[2] - d[1]))[:20]:
            print(f"{rel:34s} {ann:10d} {real:8d} {real - ann:+7d}")
        mean = sum(r - a for _, a, r in deltas) / len(deltas)
        print(f"\nmean signed delta over mismatches: {mean:+.1f}")
        print("A systematic one-sided delta means the clips were TRIMMED, not "
              "merely re-encoded -- the fraction mapping in "
              "core/data/dada.py:296 is then WRONG. See plan section 3.")


if __name__ == "__main__":
    main()

Writing /content/check_p1.py


In [25]:
%%bash
python /content/check_p1.py \
  --clips /content/probe_clips.json \
  --root "$PROBE" \
  --template 'DADA2000/{t}/{v:03d}/images' \
  --tolerance 2

checked   30   missing on disk 0
matched within +-2: 29/30 (96.7%)

clip                               annotation  on disk   delta
DADA2000/36/002/images                    345      342      -3

mean signed delta over mismatches: -3.0
A systematic one-sided delta means the clips were TRIMMED, not merely re-encoded -- the fraction mapping in core/data/dada.py:296 is then WRONG. See plan section 3.


In [28]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 36.8 MB/s eta 0:00:00


In [29]:
%%bash
cd /content/drive/MyDrive/Thesis/kat-vad
python - <<'PY'
import json, itertools
from pathlib import Path
from core.tools.extract_clip_features import preprocess_frames
from core.data.video_io import list_frame_images, read_images

root = Path('/content/probe')
rows = json.loads(Path('/content/probe_clips.json').read_text())
ok = bad = 0
for r in itertools.islice(rows, 5):
    folder = root / f"DADA2000/{r['type']}/{r['video']:03d}/images"
    if not folder.is_dir():
        print('skip (layout)', folder); continue
    paths = list_frame_images(folder)[:8]
    try:
        frames = read_images(paths)                   # (N, H, W, 3) uint8
        batch = preprocess_frames(frames, center_crop=False)   # <-- _ncc, see below
        print(f"{folder.name}: {len(paths)} frames, native {frames.shape[1:3]} "
              f"-> {tuple(batch.shape)} {batch.dtype} "
              f"range[{batch.min():.3f},{batch.max():.3f}]")
        ok += 1
    except Exception as exc:                      # probe only; report, don't mask
        print('FAIL', folder, type(exc).__name__, exc); bad += 1
print(f"\nP3: {ok} ok, {bad} failed")
PY

images: 8 frames, native (660, 1584) -> (8, 3, 224, 224) torch.float32 range[-1.748,2.146]
images: 8 frames, native (660, 1584) -> (8, 3, 224, 224) torch.float32 range[-1.792,1.986]
images: 8 frames, native (660, 1584) -> (8, 3, 224, 224) torch.float32 range[-1.792,2.146]
images: 8 frames, native (660, 1584) -> (8, 3, 224, 224) torch.float32 range[-1.792,2.146]
images: 8 frames, native (660, 1584) -> (8, 3, 224, 224) torch.float32 range[-1.792,2.116]

P3: 5 ok, 0 failed


In [30]:
%%bash
mkdir -p "$KATVAD_OUTPUT_ROOT/EDA/DADA2000Origin"
cp /content/probe_clips.json "$KATVAD_OUTPUT_ROOT/EDA/DADA2000Origin/"